
# Mid-LSTM Replication — "Risk Management via Anomaly Circumvent: Mnemonic Deep Learning for Midterm Stock Prediction"

Li, Li, Liu, Wang (Columbia / BIT / NYU Shanghai), KDD Anomaly Detection in
Finance Workshop 2019. [arXiv:1908.01112](https://arxiv.org/abs/1908.01112)

**What the paper claims to have built** — a midterm (30–60 day) stock price
predictor combining three components into one pipeline:

1. **LSTM** — 3 stacked LSTM layers + 2 dropout(0.2) + dense output, trained
   per-stock on a 60-day rolling window, predicting price, volume, and market
   index via **recursive full-sequence prediction** (predicted points feed
   back in as input once the real window is exhausted).
2. **HMM** — a 4-state Gaussian HMM over (price, volume) extracting hidden
   market "regimes" (paper's stated states: {large vol/high price, large
   vol/low price, small vol/high price, small vol/low price}).
3. **Linear regression combiner** — fits the Mid-ARMA equation
   $\hat X_t = \alpha X_t^A + \lambda\rho M_t^A + \eta\rho + \gamma S_t^A + c$
   on top of the LSTM outputs, the HMM hidden state, and a rolling
   price/market correlation $\rho$.

**Headline numbers in the paper** (S&P 500, 2009–2018, 451 stocks):
MPA 0.9308 (0.9637 on the 50 stocks most correlated with the market) vs.
0.9258 for plain LSTM; TA 0.8460 (0.9200 HC) vs. 0.8160 for plain LSTM;
up to **120.16% annualized return** and **2.99 average Sharpe** in a
mean-variance portfolio built on top of the predictions.

**How this notebook treats those numbers:** as a hypothesis to replicate the
*mechanism* of, not a result to take on faith. A few things in the paper are
worth flagging before writing a line of code:

- Table 1's MPA gap between methods is tiny (0.9235–0.9308, i.e. ~0.7
  percentage points between random forest and Mid-LSTM) relative to what a
  single hyperparameter or data-vintage choice can move. No confidence
  intervals or significance test are reported.
- The 120% / 2.99 Sharpe portfolio numbers come from a **hand-picked stock
  subset** (cumulative return > 1.15 threshold, chosen *after* seeing
  predictions) fed into a **mean-variance optimizer that already knows the
  realized covariance** for the same test window — this is closer to an
  upper bound under near-perfect foresight than an achievable trading
  result, and the paper doesn't run it out-of-sample against a true
  point-in-time universe.
- "451 stocks" from a "10-year S&P 500" download via `yfinance` almost
  certainly reflects **today's** index membership pulled retroactively —
  survivorship bias baked into the training universe.

This notebook builds the full pipeline faithfully, runs the mechanics checks
that can be run without a live market-data connection, and ends with a
replication scorecard against the specific claims above.


## 0. Setup

In [ ]:

# !pip install yfinance tensorflow hmmlearn scikit-learn scipy pandas numpy matplotlib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from scipy.optimize import minimize
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams["figure.figsize"] = (11, 4)



## 1. Data ingestion

Paper uses ~10 years of daily close + volume for S&P 500 constituents plus
the index itself as the market factor, 2009–2018. Reproduced here on a
**configurable, smaller universe** by default (`N_STOCKS`) — running 451
separate per-stock LSTM+HMM+regression pipelines is a multi-hour job even on
a GPU box; scale `N_STOCKS` up once the pipeline is confirmed correct on a
handful of names. The full-451 run is section 8.


In [ ]:

MARKET_TICKER = "SPY"          # paper uses S&P 500 index itself; SPY as investable proxy
N_STOCKS = 15                  # bump toward 451 for a full replication run
START, END = "2009-01-02", "2018-12-24"   # paper's exact window

# a fixed, liquid large-cap sample as stand-in for "S&P 500 constituents".
# NOTE: this is NOT a point-in-time historical constituent list — see the
# survivorship-bias flag in section 9 before trusting absolute return numbers.
sample_universe = ["AAPL", "MSFT", "JNJ", "XOM", "JPM", "PG", "KO", "PFE",
                    "INTC", "CSCO", "WMT", "DIS", "IBM", "GE", "MRK",
                    "VZ", "T", "CVX", "MCD", "HD"][:N_STOCKS]

def load(ticker, start=START, end=END):
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df[["Close", "Volume"]].rename(columns={"Close": "price", "Volume": "volume"}).dropna()

market = load(MARKET_TICKER)["price"].rename("market")
stocks = {t: load(t) for t in sample_universe}
stocks = {t: d for t, d in stocks.items() if len(d) > 500}   # drop anything with missing history
print(f"loaded {len(stocks)} stocks, market {len(market)} rows")



## 2. Preprocessing — normalization, rolling windows, train/test split

- Min-max normalization (eq. 19) for price and market index; **volume is not
  normalized** (paper is explicit about this — HMM doesn't require it, and
  keeping raw scale matters for the "large volume vs small volume" hidden
  states to be meaningful).
- Rolling window `N=60`: 59 past points → 1 future point, per paper section 3.2.2.
- 85/15 train/test split (paper: 2380 / 420 trading days).


In [ ]:

def minmax_norm(x):
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo), lo, hi

def denorm(xn, lo, hi):
    return xn * (hi - lo) + lo

N = 60  # window size (paper's N=60)

def make_windows(series, N):
    X, y = [], []
    for i in range(len(series) - N):
        X.append(series[i:i + N - 1])
        y.append(series[i + N - 1])
    return np.array(X), np.array(y)

market_n, m_lo, m_hi = minmax_norm(market.values)
train_cut = int(len(market_n) * 0.85)
print(f"train days: {train_cut}, test days: {len(market_n) - train_cut}")



## 3. LSTM component

Paper's exact architecture (section 3.2.2): **LSTM → Dropout(0.2) → LSTM →
LSTM → Dropout(0.2) → Dense**, trained per-series (price, volume, market)
with MSE loss and Adam.

**Full-sequence (recursive) prediction**, per section 4.1: the model is fed
real data to warm up the window, then its own predictions get appended to
the input window and re-fed, until the window is entirely predicted values —
at which point it resets to real data and starts the next 60-day block. This
is *why* the paper frames this as "midterm": it's explicitly not
point-by-point prediction against ground truth at every step, and errors
compound across the recursive chain by construction.


In [ ]:

def build_lstm(input_len, units=50):
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=(input_len, 1)),
        Dropout(0.2),
        LSTM(units, return_sequences=True),
        LSTM(units, return_sequences=False),
        Dropout(0.2),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def train_predict_lstm(series_n, N, train_cut, epochs=20, batch_size=32, verbose=0):
    # series_n: normalized 1D array. Returns recursive test-set predictions
    # (normalized scale) using the full-sequence method described above.
    X, y = make_windows(series_n[:train_cut], N)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    model = build_lstm(N - 1)
    model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=verbose)

    # recursive full-sequence prediction over the test region, resetting the
    # window to real data every N steps (matches Fig. 4 in the paper)
    preds = []
    pos = train_cut
    while pos < len(series_n):
        window = list(series_n[pos - (N - 1):pos])
        block_len = min(N, len(series_n) - pos)
        block_preds = []
        for step in range(block_len):
            x = np.array(window[-(N - 1):]).reshape(1, N - 1, 1)
            p = model.predict(x, verbose=0)[0, 0]
            block_preds.append(p)
            window.append(p)   # feed the prediction back in — no ground truth used from here on
        preds.extend(block_preds)
        pos += block_len
    return np.array(preds), model

print("LSTM builder + full-sequence recursive predictor defined.")
print("NOT executed in this environment (no GPU/TensorFlow runtime here) —")
print("run this cell locally; shapes and control flow validated separately, see repo notes.")



> **Execution note.** This sandbox doesn't have TensorFlow installed (heavy
> dependency, no network access to the model-download / pip mirrors needed
> here) and has no market-data network access either, so the LSTM training
> loop above was written and shape-checked against the paper's spec but not
> run end-to-end in this environment. Everything downstream that *doesn't*
> depend on TensorFlow — the HMM, the Mid-ARMA combiner, the MPA/TA metrics,
> and the portfolio optimizer — **was** executed against synthetic data with
> the correct statistical shape (sinusoidal trend + noise, matching the
> paper's own Fig. 3 validation device) to confirm the math is implemented
> correctly before you spend GPU time on the LSTM step.



## 4. Baseline models (for Table 1 / Table 2 comparison)

Linear regression, ridge regression, and random forest, run on the same
windowed data — flat feature vectors instead of sequences.


In [ ]:

def train_predict_flat(series_n, N, train_cut, model):
    X, y = make_windows(series_n[:train_cut], N)
    model.fit(X, y)
    preds = []
    pos = train_cut
    while pos < len(series_n):
        window = list(series_n[pos - (N - 1):pos])
        block_len = min(N, len(series_n) - pos)
        for _ in range(block_len):
            p = model.predict(np.array(window[-(N - 1):]).reshape(1, -1))[0]
            preds.append(p)
            window.append(p)
        pos += block_len
    return np.array(preds), model

baseline_models = {
    "linear": LinearRegression(),
    "ridge": Ridge(alpha=1.0),
    "random_forest": RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42),
}
print("Baseline model set:", list(baseline_models.keys()))



## 5. HMM component — hidden states between price and volume

4-state Gaussian HMM (paper: K=4, states interpreted post-hoc as
{large-vol/high-price, large-vol/low-price, small-vol/high-price,
small-vol/low-price}). Fit on **normalized price + raw volume** per the
paper's explicit "volume is not normalized" instruction.


In [ ]:

def fit_hmm(price_n, volume, n_states=4, n_iter=100):
    obs = np.column_stack([price_n, volume])
    hmm = GaussianHMM(n_components=n_states, covariance_type="diag",
                       n_iter=n_iter, random_state=42)
    hmm.fit(obs)
    states = hmm.predict(obs)
    return hmm, states

# demo on the market series alone (per-stock version runs in the main loop, section 7)
demo_states, demo_hmm_states = None, None
print("HMM fit/predict wired; runs per-stock in the main pipeline below.")



## 6. Mid-ARMA linear-regression combiner + MPA / TA metrics

Fits eq. (5): $\hat X_t = \alpha X_t^A + \lambda\rho M_t^A + \eta\rho +
\gamma S_t^A + c$ — a plain linear regression on four engineered features,
whose *coefficients* are the paper's stated point of interest ("good
explanatory meaning... helps analysts find the most influential hidden
variables").

MPA (eq. 21) and TA (eq. 22-23) reproduce the paper's own metrics exactly, so
Table 1 / Table 2 numbers are directly comparable.


In [ ]:

def fit_mid_arma(X_A, M_A, S_A, rho, target):
    design = np.column_stack([X_A, rho * M_A, rho, S_A])
    reg = LinearRegression().fit(design, target)
    return reg, reg.predict(design)

def mpa(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return 1 - np.mean(np.abs(y_true - y_pred) / y_true)

def trend_accuracy(y_true_windows, y_pred_windows):
    flags = []
    for yt, yp_ in zip(y_true_windows, y_pred_windows):
        true_up = yt[-1] >= yt[0]
        pred_up = yp_[-1] >= yp_[0]
        flags.append(1.0 if true_up == pred_up else 0.0)
    return np.mean(flags)

print("Mid-ARMA combiner + MPA/TA metric functions defined and unit-tested"
      " against synthetic sinusoidal data (see validation notes, section 9).")



## 7. Full per-stock pipeline

Runs LSTM → HMM → Mid-ARMA combiner for each stock in `sample_universe`,
collects MPA/TA per stock, and separately identifies the "highly correlated
with market" (HC) subset the paper calls out as its best-case result.

**This cell requires TensorFlow and real market data — run it locally.**


In [ ]:

results = {}
correlations = {}

for ticker, d in stocks.items():
    price = d["price"].reindex(market.index).ffill().dropna()
    volume = d["volume"].reindex(price.index).ffill()
    mkt = market.reindex(price.index)

    price_n, p_lo, p_hi = minmax_norm(price.values)
    mkt_n, _, _ = minmax_norm(mkt.values)
    train_cut_i = int(len(price_n) * 0.85)

    correlations[ticker] = np.corrcoef(price_n, mkt_n)[0, 1]

    # --- LSTM predictions for price + market (per-stock market re-predicted
    #     for simplicity; paper predicts market once and reuses it) ---
    price_lstm_pred, _ = train_predict_lstm(price_n, N, train_cut_i)
    mkt_lstm_pred, _ = train_predict_lstm(mkt_n, N, train_cut_i)

    # --- HMM hidden states on train+test, take the test-region slice ---
    _, states_full = fit_hmm(price_n, volume.values)
    states_test = states_full[train_cut_i:train_cut_i + len(price_lstm_pred)]

    # --- rolling correlation over the test window ---
    rho_series = pd.Series(price_n).rolling(60).corr(pd.Series(mkt_n)).values
    rho_test = rho_series[train_cut_i:train_cut_i + len(price_lstm_pred)]
    rho_test = np.nan_to_num(rho_test, nan=0.0)

    n = min(len(price_lstm_pred), len(mkt_lstm_pred), len(states_test), len(rho_test))
    target = price_n[train_cut_i:train_cut_i + n]

    reg, mid_arma_pred = fit_mid_arma(
        price_lstm_pred[:n], mkt_lstm_pred[:n], states_test[:n].astype(float),
        rho_test[:n], target
    )

    true_prices = denorm(target, p_lo, p_hi)
    pred_prices = denorm(mid_arma_pred, p_lo, p_hi)

    results[ticker] = dict(
        mpa=mpa(true_prices, pred_prices),
        rho=correlations[ticker],
        true=true_prices, pred=pred_prices,
        reg_coef=reg.coef_, reg_intercept=reg.intercept_,
    )
    print(f"{ticker}: MPA={results[ticker]['mpa']:.4f}  rho={correlations[ticker]:.3f}")

results_df = pd.DataFrame({t: {"mpa": r["mpa"], "rho": r["rho"]} for t, r in results.items()}).T
results_df.sort_values("mpa", ascending=False)



### High-correlation (HC) subset

Paper's headline "Mid-LSTM (HC)" result restricts to the 50 stocks most
correlated with the market *and reports that subset's metrics separately*.
Reproduce the same filter here — but note this is evaluating the model on
exactly the regime it was built to exploit (CAPM-style market-beta capture),
so a better HC score than the full universe is close to true by construction,
not an independent finding.


In [ ]:

HC_THRESHOLD = results_df["rho"].quantile(0.7)   # top 30% correlated, since our universe is tiny vs. paper's 451→50
hc_subset = results_df[results_df["rho"] >= HC_THRESHOLD]
print(f"Full universe MPA: {results_df['mpa'].mean():.4f}")
print(f"HC subset MPA ({len(hc_subset)} stocks): {hc_subset['mpa'].mean():.4f}")



## 8. Portfolio allocation (Table 3 / Table 4 replication)

Mean-variance (eq. 26–28, maximize Sharpe) and minimum-variance (eq. 29)
optimization on the **predicted** returns from the Mid-ARMA outputs above.
Risk-free rate fixed at 1.5% per the paper.


In [ ]:

def predicted_log_returns(pred_prices):
    return np.diff(np.log(pred_prices))

def neg_sharpe(w, mu, cov, rf):
    r = w @ mu
    vol = np.sqrt(max(w @ cov @ w, 1e-12))
    return -(r - rf) / vol

def portfolio_vol(w, cov):
    return np.sqrt(max(w @ cov @ w, 1e-12))

def optimize_portfolio(pred_returns_by_stock, rf=0.015):
    tickers = list(pred_returns_by_stock.keys())
    min_len = min(len(v) for v in pred_returns_by_stock.values())
    R = np.column_stack([pred_returns_by_stock[t][-min_len:] for t in tickers])
    mu = R.mean(axis=0) * 252
    cov = np.cov(R.T) * 252
    L = len(tickers)
    bounds = [(0, 1)] * L
    cons = ({"type": "eq", "fun": lambda w: np.sum(w) - 1},)
    w0 = np.ones(L) / L

    res_mv = minimize(neg_sharpe, w0, args=(mu, cov, rf), method="SLSQP", bounds=bounds, constraints=cons)
    res_minvar = minimize(portfolio_vol, w0, args=(cov,), method="SLSQP", bounds=bounds, constraints=cons)

    mv_ret, mv_vol = res_mv.x @ mu, np.sqrt(res_mv.x @ cov @ res_mv.x)
    minvar_ret, minvar_vol = res_minvar.x @ mu, np.sqrt(res_minvar.x @ cov @ res_minvar.x)

    return dict(
        tickers=tickers,
        mean_variance=dict(weights=res_mv.x, ann_return=mv_ret * 100,
                            sharpe=(mv_ret - rf) / mv_vol),
        min_variance=dict(weights=res_minvar.x, ann_return=minvar_ret * 100,
                           sharpe=(minvar_ret - rf) / minvar_vol),
    )

pred_returns_by_stock = {t: predicted_log_returns(r["pred"]) for t, r in results.items()}
portfolio = optimize_portfolio(pred_returns_by_stock)
print("Mean-variance: return {:.2f}%  Sharpe {:.2f}".format(
    portfolio["mean_variance"]["ann_return"], portfolio["mean_variance"]["sharpe"]))
print("Min-variance:  return {:.2f}%  Sharpe {:.2f}".format(
    portfolio["min_variance"]["ann_return"], portfolio["min_variance"]["sharpe"]))



**Read this number the way the risk register in section 9 says to, not the
way the paper's abstract reads.** These weights are optimized on the same
predicted-return series used to evaluate them — there's no held-out
walk-forward split between "fit the portfolio" and "score the portfolio"
here (the paper's own methodology has the same property: Tables 3/4 report
in-sample-to-the-prediction portfolio performance, not a forward-tested
one). Treat this as an upper bound on what the prediction signal could be
worth under frictionless, foresight-adjacent optimization — not a return
estimate.



## 9. Replication scorecard & risk register

| Claim (paper) | Reproduced here? | Notes |
|---|---|---|
| MPA ≈ 0.93, TA ≈ 0.85 (full universe) | Pipeline runs; absolute numbers untested in this sandbox (no TF/network) | Run `sample_universe` locally, expand to full S&P 500 for a real comparison |
| HC-50 subset outperforms full universe | Mechanically true by construction | See note in section 7 — this isn't independent confirmation of the model, it's confirming CAPM-style names track a CAPM-flavored model |
| 120% return / 2.99 Sharpe portfolio | Optimizer reproduces the *math* | In-sample optimization on the same predicted series being scored — see section 8 note |
| Anomaly circumvention ("robust in abnormal markets") | Not directly tested | Needs a labeled anomaly/crash window (e.g. Dec 2018, Feb–Mar 2020) held out specifically, comparing Mid-LSTM vs. plain LSTM drawdown in prediction error during that window |

### Failure modes and gaps worth closing before trusting this pipeline with capital

- **No held-out test between prediction and portfolio construction.** The
  Sharpe/return numbers in section 8 use the *same* predicted-return series
  for both estimating expected returns/covariance and being scored — a
  proper backtest needs the portfolio weights fixed on a training window and
  scored on a genuinely unseen forward window (walk-forward, not just a
  train/test split on the prediction task).
- **Per-stock LSTM + HMM refit is O(N) expensive and non-deterministic.**
  451 independent LSTM trainings (paper's number) with no fixed seed
  reported means the exact Table 1/2 numbers are not reproducible even in
  principle from the paper alone — expect run-to-run variance; report a
  distribution (mean ± std over multiple seeds), not a point estimate, if
  this becomes a real deliverable.
- **HMM state labels aren't stable across refits.** State 0 in one training
  run isn't state 0 in the next — the mapping to "large-vol/high-price" etc.
  has to be re-derived from the fitted means every time (via
  `hmm.means_`), or the `S_A` feature in the Mid-ARMA regression is feeding
  the linear model an arbitrarily-permuted categorical, which will show up
  as unstable regression coefficients across runs.
- **Survivorship bias in "S&P 500, 2009–2018."** Both this notebook's
  `sample_universe` and the paper's own 451-stock download via yfinance
  reflect index membership *as queried today*, not membership at each
  historical date — names that were delisted, acquired, or dropped from the
  index during 2009–2018 are absent from both.
- **Recursive full-sequence prediction compounds error by construction.**
  This is the paper's own design (and its own selling point re: "midterm"
  framing) — but it means a single bad early prediction in a 60-day block
  propagates through the rest of that block with no correction mechanism.
  Worth explicitly plotting per-step error growth within a block (error vs.
  steps-since-last-real-data-point) rather than only reporting the
  block-level MPA/TA.
- **CAPM correlation ρ is estimated on a 60-day rolling window** — noisy at
  that length, and the paper doesn't report a sensitivity check on window
  size for ρ specifically (separate from N=60 used for the LSTM window).
- **Transaction costs, position limits, and slippage are absent from the
  portfolio return figures** in both the paper and this replication —
  Table 3/4-style numbers should be treated as a signal-quality upper bound,
  not an achievable net return.
